# 23.1 AWS 数据科学栈:对象存储与无服务器 / AWS for Data Science

**中文**:今天几乎没有公司自己买机器搭机房了——数据、训练、部署全在**云**上。而 **AWS(Amazon Web Services)** 是市占率第一的云,数据科学岗几乎必然要用到它。云服务多如牛毛(几百个),但对数据科学家,真正核心的就那么几个,而且它们背后是**两个改变一切的思想**:①**对象存储(S3)+ 存算分离** —— 数据无限便宜地存在一个地方,计算按需拉起、用完释放;②**无服务器(serverless, Lambda)** —— 你只写函数,云负责在事件发生时运行它、自动扩缩、按次计费,**你从此不用管服务器**。本节从零实现一个 mini-S3 和 mini-Lambda,亲手搭一条"图片上传→自动生成缩略图"的事件驱动流水线,让你彻底理解云的这两大基石,再给出 AWS 数据科学服务全景图。
**English**: Today almost no company buys machines and builds server rooms — data, training, and deployment all live in the **cloud**. And **AWS (Amazon Web Services)** is the #1 cloud by market share, nearly unavoidable in data science roles. Cloud services are countless (hundreds), but for data scientists only a handful truly matter, and behind them are **two ideas that changed everything**: ① **object storage (S3) + storage-compute separation** — data lives infinitely cheaply in one place, compute spins up on demand and releases when done; ② **serverless (Lambda)** — you write only a function, and the cloud runs it when events occur, auto-scales, and bills per-invocation, so **you never manage servers**. This section builds a mini-S3 and mini-Lambda from scratch, hand-assembling an "image upload → auto thumbnail" event-driven pipeline, so you deeply understand these two cloud cornerstones, then gives the AWS data-science service map.

---

**中文**:**两个改变一切的云思想**:
**English**: **Two cloud ideas that changed everything**:
- **中文**:**对象存储(S3)= 无限大的 key→bytes 键值仓库**。它**不是文件系统**——没有真正的文件夹,`data/2024/x.parquet` 只是一个看起来像路径的 key。特点:①**近乎无限、极便宜**(几分钱/GB/月)②**11 个 9 的持久性**(几乎不丢数据)③**存算分离的基础**——数据存 S3,任何计算(EC2、SageMaker、Athena)按需读取。**现代数据湖/湖仓(Part 21)的底座就是 S3**。
  **Object storage (S3) = an infinite key→bytes store**. It is **not a filesystem** — no real folders; `data/2024/x.parquet` is just a key that looks like a path. Features: ① **near-infinite, very cheap** (cents/GB/month) ② **11 nines of durability** (virtually never loses data) ③ **the basis of storage-compute separation** — data lives in S3, and any compute (EC2, SageMaker, Athena) reads on demand. **Modern data lakes/lakehouses (Part 21) are built on S3.**
- **中文**:**无服务器(Lambda)= 只写函数,云来运行**。传统上你要租一台服务器(EC2)、让它 24 小时开着等请求。Lambda 反过来:你只上传一个函数 + 声明"什么事件触发它"(S3 有新文件、API 被调用、定时器),云在事件发生时**才**运行它,**自动扩缩(一次来 1000 个事件就并行跑 1000 个)、按运行次数和时长计费、空闲时零成本**。你彻底不用管服务器的存在。
  **Serverless (Lambda) = write only a function, the cloud runs it**. Traditionally you rent a server (EC2) and keep it running 24/7 waiting for requests. Lambda inverts this: you upload just a function + declare "what event triggers it" (a new S3 file, an API call, a timer), and the cloud runs it **only** when the event occurs, **auto-scaling (1000 events at once → 1000 parallel runs), billing per invocation and duration, zero cost when idle**. You never think about servers.

**中文**:**AWS 数据科学核心服务全景**(面试要能对上号):
**English**: **AWS data-science core service map** (know the mapping for interviews):
- **中文**:**存储**:S3(对象存储/数据湖)。**计算**:EC2(虚拟机)、Lambda(无服务器函数)。**数仓/查询**:Athena(直接用 SQL 查 S3 上的 Parquet,serverless,按扫描量计费)、Redshift(数仓)。**ML 平台**:SageMaker(训练/部署/notebook 一体)。**数据管道**:Glue(ETL)、EMR(托管 Spark)、Kinesis(流)。**编排**:Step Functions / MWAA(托管 Airflow)。
  **Storage**: S3 (object storage/data lake). **Compute**: EC2 (virtual machines), Lambda (serverless functions). **Warehouse/query**: Athena (query Parquet on S3 directly with SQL, serverless, billed per bytes scanned), Redshift (warehouse). **ML platform**: SageMaker (training/deployment/notebooks in one). **Data pipelines**: Glue (ETL), EMR (managed Spark), Kinesis (streaming). **Orchestration**: Step Functions / MWAA (managed Airflow).

> 💡 **面试速查 / Interview cheat-sheet（★★ 云基础必考）**
> **中文**:**AWS 数据科学栈**:**S3**(对象存储=无限便宜的 key→bytes, 数据湖底座, **不是文件系统**, 存算分离基础)、**EC2**(虚拟机)、**Lambda**(无服务器函数, 事件触发/自动扩缩/按次计费/空闲零成本)、**Athena**(serverless SQL 查 S3 Parquet, 按扫描字节计费→用分区+列式省钱)、**Redshift**(数仓)、**SageMaker**(ML 训练/部署/notebook)、**Glue**(ETL)、**EMR**(托管 Spark)、**Kinesis**(流)。**两大思想**:①**存算分离**(数据在 S3, 计算按需拉起/释放→弹性省钱, vs 传统绑定)②**无服务器**(不养服务器, 事件驱动, 自动扩缩)。**成本模型**:S3 按存储量、Athena 按扫描量、Lambda 按调用+时长、EC2 按时长(Spot 便宜/按需贵/预留最省)。**IAM** 管权限(最小权限原则)。**存储分层**:S3 Standard→IA→Glacier(冷数据省钱)。面试金句:*"AWS 数据科学核心是 S3 对象存储做数据湖(存算分离基础, 不是文件系统)、按需 EC2/Lambda 计算、Athena serverless 查 S3、SageMaker 做 ML 全流程; 两大思想是存算分离(数据在 S3 计算按需拉起省钱)和无服务器(不养服务器、事件驱动、自动扩缩); 成本靠分区/列式(减 Athena 扫描)、Spot 实例、存储分层优化。"*
> **English**: **AWS data-science stack**: **S3** (object storage = infinitely cheap key→bytes, data-lake foundation, **not a filesystem**, basis of storage-compute separation), **EC2** (VMs), **Lambda** (serverless functions, event-triggered/auto-scaling/per-invocation billing/zero idle cost), **Athena** (serverless SQL on S3 Parquet, billed per bytes scanned → use partitions + columnar to save), **Redshift** (warehouse), **SageMaker** (ML training/deployment/notebooks), **Glue** (ETL), **EMR** (managed Spark), **Kinesis** (streaming). **Two ideas**: ① **storage-compute separation** (data in S3, compute spun up/released on demand → elastic and cheap, vs traditional coupling) ② **serverless** (no servers to manage, event-driven, auto-scaling). **Cost model**: S3 per storage, Athena per bytes scanned, Lambda per invocation + duration, EC2 per hour (Spot cheap / on-demand pricey / reserved cheapest). **IAM** manages permissions (least-privilege). **Storage tiering**: S3 Standard→IA→Glacier (cold data saves money). Interview line: *"AWS's data-science core is S3 object storage as the data lake (storage-compute separation basis, not a filesystem), on-demand EC2/Lambda compute, Athena serverless queries on S3, SageMaker for the ML lifecycle; the two ideas are storage-compute separation (data in S3, compute on demand saves money) and serverless (no servers, event-driven, auto-scaling); optimize cost with partitions/columnar (fewer Athena scans), Spot instances, and storage tiering."*


In [ ]:

# ============================================================
# 从零实现 mini-S3(对象存储)+ mini-Lambda(无服务器事件触发)/ mini-S3 + mini-Lambda from scratch
# 中文:对象存储=扁平的 key→bytes 键值仓库(按 bucket 分组), 不是文件系统。无服务器=注册一个函数,
#      在事件(如新对象上传)发生时自动运行——不用养服务器、不用轮询。
# English: object storage = a flat key→bytes store (grouped by bucket), not a filesystem. Serverless = register a
#      function that runs automatically on an event (e.g. a new object upload) — no server to run, no polling.
# ============================================================
import fnmatch
class MiniEvents:                                          # 事件系统(对标 S3 事件通知)/ event system (like S3 events)
    def __init__(self): self.triggers=[]
    def on_put(self, bucket, pattern, fn): self.triggers.append((bucket, pattern, fn))  # 注册触发器 / register a trigger
    def dispatch(self, bucket, key):
        for b, pat, fn in self.triggers:
            if b==bucket and fnmatch.fnmatch(key, pat): fn(bucket, key)   # 匹配就运行函数(=Lambda 被触发)/ run fn (Lambda fires)
events=MiniEvents()

class MiniS3:                                              # 对象存储:bucket -> {key: bytes} / object store
    def __init__(self): self.buckets={}
    def create_bucket(self, b): self.buckets.setdefault(b, {})
    def put(self, b, key, data):
        self.buckets[b][key]=data; events.dispatch(b, key)               # put 触发事件(S3→Lambda)/ put triggers events
    def get(self, b, key): return self.buckets[b][key]
    def list(self, b, prefix=""): return [k for k in self.buckets[b] if k.startswith(prefix)]

s3=MiniS3(); s3.create_bucket("raw-data"); s3.create_bucket("thumbnails")
# 一个"Lambda":只要有 .jpg 落到 raw-data, 自动生成缩略图存到 thumbnails——事件驱动, 无需服务器 / an event-driven Lambda
processed=[]
def make_thumbnail(bucket, key):
    img=s3.get(bucket, key)
    s3.put("thumbnails", key.replace(".jpg","_thumb.jpg"), f"thumbnail({img})")
    processed.append(key)
events.on_put("raw-data", "*.jpg", make_thumbnail)                       # 声明:raw-data 里的 *.jpg 触发这个函数 / trigger declaration

for name in ["cat.jpg","dog.jpg","notes.txt"]:                          # 上传三个对象 / upload three objects
    s3.put("raw-data", name, f"<bytes of {name}>")                       # 上传即自动触发 Lambda / upload auto-fires Lambda
print("raw-data 桶的对象 / objects:", s3.list("raw-data"))
print("Lambda 自动处理的图片 / auto-processed by Lambda:", processed, "  (notes.txt 不匹配 *.jpg → 未触发)")
print("thumbnails 桶(全部由 Lambda 生成)/ thumbnails (all created by Lambda):", s3.list("thumbnails"))
print("→ 没有服务器在运行/轮询; 数据一到达, 无服务器函数就被事件自动触发处理——这就是 serverless")


In [ ]:

# ============================================================
# 存算分离 + 云成本模型直觉 / storage-compute separation + cloud cost intuition
# 中文:云的省钱哲学=存(便宜, 一直存)和算(贵, 按需拉起用完释放)解耦。对比"传统 24 小时开机" vs "按需计算"的成本。
# English: the cloud's cost philosophy = decouple storage (cheap, always on) from compute (pricey, spin up on demand, release).
#      Compare "traditional always-on server" vs "on-demand compute" cost.
# ============================================================
import matplotlib.pyplot as plt
# 场景:每天只需 1 小时跑一个训练任务, 其余时间闲置 / scenario: 1 hour of training per day, idle otherwise
hours_per_day_used=1; days=30
ec2_hourly=1.0                                            # 一台 GPU 实例约 $1/小时(示意)/ ~$1/hr GPU instance (illustrative)
cost_always_on = ec2_hourly*24*days                      # 传统:24 小时开机 / always-on 24/7
cost_on_demand = ec2_hourly*hours_per_day_used*days      # 云:只在用时计费 / on-demand: pay only when used
cost_spot      = cost_on_demand*0.3                       # Spot 实例约 3 折(可被回收)/ Spot ~30% (interruptible)
print(f"每天只用 1 小时算力, 连续 30 天的成本 / cost of 1 hr/day compute over 30 days:")
print(f"  传统 24h 开机 always-on : ${cost_always_on:.0f}")
print(f"  云按需 on-demand        : ${cost_on_demand:.0f}   ({cost_always_on/cost_on_demand:.0f}x 便宜, 只为实际使用付费)")
print(f"  云 Spot 抢占实例        : ${cost_spot:.0f}   (再省 70%, 代价是可能被中断)")
fig,ax=plt.subplots(1,2,figsize=(14,5))
ax[0].bar(["传统\n24h开机","云\n按需","云\nSpot"],[cost_always_on,cost_on_demand,cost_spot],color=["#C44E52","#55A868","#4C72B0"])
for i,v in enumerate([cost_always_on,cost_on_demand,cost_spot]): ax[0].text(i,v+5,f"${v:.0f}",ha="center",fontsize=11,weight="bold")
ax[0].set_ylabel("30天成本 $"); ax[0].set_title("存算分离:只为实际用的计算付费")
# AWS 服务地图 / service map
ax[1].axis("off"); ax[1].set_title("AWS 数据科学服务地图",fontsize=12,weight="bold")
rows=[("存储 Storage","S3(对象存储/数据湖底座)","#4C72B0"),
      ("计算 Compute","EC2(虚拟机) · Lambda(无服务器)","#55A868"),
      ("查询/数仓 Query","Athena(查S3, 按扫描计费) · Redshift","#DD8452"),
      ("ML 平台","SageMaker(训练/部署/notebook)","#9467BD"),
      ("管道/编排","Glue · EMR(Spark) · Kinesis · Step Functions","#C44E52")]
for i,(cat,svc,c) in enumerate(rows):
    ax[1].add_patch(plt.Rectangle((0.05,0.8-i*0.16),0.9,0.13,fc=c,alpha=0.2,ec=c,transform=ax[1].transAxes))
    ax[1].text(0.08,0.87-i*0.16,cat,fontsize=9,weight="bold",color=c,transform=ax[1].transAxes)
    ax[1].text(0.08,0.815-i*0.16,svc,fontsize=8,transform=ax[1].transAxes)
plt.tight_layout(); plt.savefig("/tmp/cloud01_viz.png",dpi=80); plt.show()
print("左:按需计算只为实际使用付费(比 24h 开机省几十倍); 右:AWS 数据科学核心服务地图")


**中文**:诚实解读:
**English**: Honest takeaways:

**中文**:
1. **云的两大基石思想比记服务名重要一万倍:存算分离 + 无服务器**:①**存算分离**——传统上,数据和计算绑在一台机器上(数据在哪台机器,就在哪台算),扩容要整机一起扩,浪费严重。云把它们**彻底解耦**:数据无限便宜地躺在 S3 里(一直存,几分钱/GB),计算(EC2/SageMaker/Athena)在需要时**按需拉起、用完立即释放**。我们的成本演示一针见血:每天只用 1 小时算力,传统 24 小时开机要 $720,按需只要 $30——**省 24 倍**,而这仅仅因为"不用时不付钱"。②**无服务器**——你连"拉起一台机器"都不用管了,只写函数、声明触发事件,云负责运行、扩缩、计费。我们的图片流水线里,没有任何服务器在运行或轮询,数据一到 S3,Lambda 就被事件自动触发。理解了这两点,几百个 AWS 服务不过是它们的不同包装。
2. **对象存储不是文件系统——这个认知差异会咬人**:S3 看起来有"文件夹"(`data/2024/x.parquet`),但那只是一个**扁平的 key**,`/` 只是 key 里的普通字符,没有真正的目录结构。这带来实际影响:①**列举一个"目录"下的对象**其实是按前缀过滤,海量对象时要注意;②**没有"重命名文件夹"这种原子操作**(要逐个复制+删除)——这正是 Part 21 湖仓事务日志要解决的问题之一;③但换来的是**近乎无限的扩展和 11 个 9 的持久性**。把 S3 当"无限大、超可靠、但只能整存整取的 key-value 仓库"来理解,而不是"云上的硬盘",才不会踩坑。
3. **诚实的边界:云省钱是"用对了才省",用错了是账单刺客**。①**云不是自动更便宜**——它的价值是**弹性**(按需伸缩)和**免运维**(托管服务),但如果你把一台大 EC2 24 小时开着不关、或者用 Athena 反复全表扫描不做分区,账单会比自建机房还贵。**成本优化是一门真功夫**:Athena/BigQuery 按扫描量计费→用 Parquet 分区+列裁剪(接 21.8);计算用 Spot 实例(可省 70%,代价是可能被回收,适合可容错的批处理);存储做分层(热数据 S3 Standard、冷数据 Glacier);及时关掉不用的资源。②**厂商锁定(vendor lock-in)** 是长期风险——深度用某家的专有服务(如 Redshift、SageMaker 的专有功能),迁移成本很高;所以很多团队偏好**开放格式(Parquet/Iceberg)+ 可移植的开源工具(Spark/DuckDB)**,把数据和逻辑留在自己手里。③**别为了"上云"而上云**:小项目单机(Part 21 的 Polars/DuckDB)往往又快又省,云的价值在规模、弹性、协作、免运维。**结论:AWS(及所有云)的精髓是存算分离 + 无服务器带来的弹性和免运维; 理解 S3 对象存储是数据湖底座、按需计算只为使用付费, 比背服务名更重要; 但云省钱靠正确的成本优化(分区/Spot/分层)和警惕厂商锁定, 用错了云比自建更贵。**

**English**:
1. **The cloud's two cornerstone ideas matter ten-thousand times more than memorizing service names: storage-compute separation + serverless**: ① **Storage-compute separation** — traditionally data and compute are bound to one machine (compute happens where the data sits), and scaling means scaling the whole machine, hugely wasteful. The cloud **fully decouples** them: data lies infinitely cheaply in S3 (always on, cents/GB), and compute (EC2/SageMaker/Athena) **spins up on demand and releases immediately when done**. Our cost demo cuts to the point: for 1 hour of daily compute, an always-on 24/7 server costs $720 while on-demand costs $30 — **24x cheaper**, simply because "you don't pay when not using." ② **Serverless** — you don't even manage "spinning up a machine"; you write only a function and declare its triggering event, and the cloud runs, scales, and bills it. In our image pipeline, no server runs or polls; data hits S3 and Lambda is auto-triggered by the event. Grasp these two and the hundreds of AWS services are just different packagings.
2. **Object storage is not a filesystem — this misunderstanding bites**: S3 looks like it has "folders" (`data/2024/x.parquet`), but that's just a **flat key**, with `/` an ordinary character in the key and no real directory structure. Practical implications: ① **listing objects under a "directory"** is actually prefix filtering, needing care at massive scale; ② **there's no atomic "rename folder"** (you copy + delete each object) — one of the problems Part 21's lakehouse transaction log solves; ③ but in return you get **near-infinite scale and 11 nines of durability**. Understand S3 as "an infinitely large, super-reliable key-value store that only stores/retrieves whole objects," not "a hard drive in the cloud," and you won't trip.
3. **Honest limits: the cloud saves money only when used right — misused, it's a bill assassin**. ① **The cloud isn't automatically cheaper** — its value is **elasticity** (scale on demand) and **no ops** (managed services), but leaving a big EC2 running 24/7, or repeatedly full-scanning with Athena without partitioning, costs more than a self-hosted server room. **Cost optimization is a real skill**: Athena/BigQuery bill per bytes scanned → use Parquet partitioning + column pruning (per 21.8); compute uses Spot instances (save 70%, at the cost of possible reclamation, good for fault-tolerant batch); tier storage (hot in S3 Standard, cold in Glacier); shut down idle resources promptly. ② **Vendor lock-in** is a long-term risk — deep use of a vendor's proprietary services (Redshift, SageMaker proprietary features) makes migration costly; so many teams prefer **open formats (Parquet/Iceberg) + portable open-source tools (Spark/DuckDB)**, keeping data and logic in their own hands. ③ **Don't cloud for cloud's sake**: small projects on a single machine (Part 21's Polars/DuckDB) are often faster and cheaper; the cloud's value is scale, elasticity, collaboration, and no ops. **Conclusion: the essence of AWS (and every cloud) is the elasticity and no-ops from storage-compute separation + serverless; understanding S3 object storage as the data-lake foundation and on-demand compute paying only for usage matters more than memorizing service names; but cloud savings depend on correct cost optimization (partitioning/Spot/tiering) and wariness of vendor lock-in — misused, the cloud is pricier than self-hosting.**

> 💼 **实战视角 / Practical angle**
> **中文**:AWS 数据科学落地:①**数据湖用 S3**(存 Parquet, 分区组织)+ **Athena** serverless 查询(分区裁剪省扫描费);②**训练/部署用 SageMaker**(托管 notebook/训练作业/端点)或自己在 EC2 上跑;③**批处理用 Spot 实例**(省 70%, 容错场景)、**无服务器用 Lambda**(轻量事件处理/API);④**ETL 用 Glue**(serverless Spark)或 EMR(重型 Spark);⑤**成本三板斧**:分区+列式(减扫描)、Spot(减计算)、存储分层(减存储);⑥**IAM 最小权限**、别把凭证写进代码(用角色);⑦**开放格式(Parquet/Iceberg)防锁定**。学习路径:先 S3+Athena 玩数据湖, 再 SageMaker 做 ML, 用 IAM 管权限。面试金句:*"AWS 数据科学核心:S3 对象存储做数据湖(存算分离基础)、EC2/Lambda 按需计算、Athena serverless 查 S3、SageMaker 做 ML 全流程; 精髓是存算分离(数据便宜地存 S3, 计算按需拉起省几十倍)和无服务器(不养服务器事件驱动); 成本靠分区/列式减 Athena 扫描、Spot 减计算、分层减存储, 并用开放格式防厂商锁定。"*
> **English**: AWS data-science in practice: ① **data lake on S3** (store Parquet, organized by partitions) + **Athena** serverless queries (partition pruning saves scan cost); ② **training/deployment via SageMaker** (managed notebooks/training jobs/endpoints) or run yourself on EC2; ③ **batch on Spot instances** (save 70%, fault-tolerant cases), **serverless via Lambda** (lightweight event processing/APIs); ④ **ETL via Glue** (serverless Spark) or EMR (heavy Spark); ⑤ **three cost levers**: partitioning + columnar (fewer scans), Spot (less compute), storage tiering (less storage); ⑥ **IAM least-privilege**, never hardcode credentials (use roles); ⑦ **open formats (Parquet/Iceberg) prevent lock-in**. Learning path: play with the data lake on S3 + Athena first, then SageMaker for ML, IAM for permissions. Interview line: *"AWS's data-science core: S3 object storage as the data lake (storage-compute separation basis), on-demand EC2/Lambda compute, Athena serverless queries on S3, SageMaker for the ML lifecycle; the essence is storage-compute separation (data cheaply in S3, compute on demand saves tens of times) and serverless (no servers, event-driven); optimize cost via partitioning/columnar to cut Athena scans, Spot to cut compute, tiering to cut storage, and use open formats to prevent vendor lock-in."*

---
### 小结 / Summary
- **中文**:AWS 数据科学核心:S3(对象存储/数据湖底座)、EC2/Lambda(按需/无服务器计算)、Athena(查S3)、SageMaker(ML)。
- **English**: AWS data-science core: S3 (object storage/data-lake foundation), EC2/Lambda (on-demand/serverless compute), Athena (query S3), SageMaker (ML).
- **中文**:两大思想:存算分离(数据在 S3 便宜存、计算按需拉起省几十倍)+ 无服务器(不养服务器、事件驱动、自动扩缩)。
- **English**: Two ideas: storage-compute separation (data cheaply in S3, compute on demand saves tens of times) + serverless (no servers, event-driven, auto-scaling).
- **中文**:S3 是 key→bytes 键值仓库(非文件系统); 云省钱靠分区/Spot/分层的成本优化, 并用开放格式防锁定。
- **English**: S3 is a key→bytes store (not a filesystem); cloud savings come from partitioning/Spot/tiering cost optimization, with open formats to prevent lock-in.
